# Tone Match Model Training

This notebook handles the training of the machine learning model used to detect text tone. 
It uses a **Logistic Regression** classifier with **TF-IDF Vectorization**.

In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

ModuleNotFoundError: No module named 'sklearn'

### Custom Text Preprocessor
We define a custom transformer to handle basic text cleaning (lowercasing and punctuation removal).

In [ ]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        processed_data = []
        for text in X:
            # Lowercase
            text = text.lower()
            # Remove punctuation (keep letters and spaces only)
            text = re.sub(r'[^a-z\s]', '', text)
            # Remove multiple spaces/newlines
            text = re.sub(r'\s+', ' ', text).strip()
            processed_data.append(text)
        return processed_data

### Data Loading and Splitting
Load the synthetic dataset and split it into training and testing sets.

In [ ]:
# Load the dataset
df = pd.read_csv("tone_dataset.csv")
X, y = df['content'], df['tone']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   
)

### Model Pipeline
Construct a pipeline that combines preprocessing, TF-IDF vectorization, and the Logistic Regression classifier.

In [ ]:
model = Pipeline([
    ("preprocessor", TextPreprocessor()),
    ("vectorizer", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        max_features=8000,
        min_df=2,          
        max_df=0.95         
    )),
    # Regularization (C=1.0) helps the model generalize well
    ("classifier", LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs"))
])

### Training and Evaluation

In [ ]:
# Train the model
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy*100:.2f}%")

In [ ]:
import joblib

# Save the trained pipeline for use in the app
joblib.dump(model, "tone_model.pkl")
print("Model saved as tone_model.pkl")